# Stage B — OLMoE local inference and expert tracing

This notebook uses the real `allenai/OLMoE-1B-7B-0924-Instruct` Hugging Face checkpoint. It first establishes a normal-generation baseline, then records the actual routed expert IDs and runtime telemetry. It does not use GGUF or framework offloading.

In [ ]:
#%pip install torch transformers accelerate safetensors huggingface_hub psutil pandas matplotlib seaborn

import gc, json, time, platform
from pathlib import Path
from collections import Counter, OrderedDict
from dataclasses import dataclass, asdict
import pandas as pd, psutil, torch
import matplotlib.pyplot as plt

assert platform.system() == 'Windows', 'Run this notebook on native Windows.'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

In [ ]:
device_id = 0
memory_fraction = 1.0
torch.cuda.set_per_process_memory_fraction(memory_fraction, device=device_id)
print(f"Set CUDA memory fraction to {memory_fraction} for device {device_id}.")

## 1. Configure the downloaded checkpoint

Change `MODEL_DIR` to the directory created by `hf download`. Keep the model in safetensors format.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = 'allenai/OLMoE-1B-7B-0924-Instruct'
# In a notebook, the working directory should be the directory containing this file.
PROJECT_DIR = Path.cwd().resolve()
MODEL_DIR = PROJECT_DIR
MAX_NEW_TOKENS = 32
PROMPT = 'Explain why loading only the experts selected by a router can reduce memory use.'

required_files = [
    MODEL_DIR / 'config.json',
    MODEL_DIR / 'model.safetensors.index.json',
    MODEL_DIR / 'tokenizer.json',
]
if not all(path.exists() for path in required_files):
    print(f'Downloading {MODEL_ID} into {MODEL_DIR}')
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )
else:
    print(f'Checkpoint already present in {MODEL_DIR}; skipping download.')

assert all(path.exists() for path in required_files), f'Incomplete checkpoint: {MODEL_DIR}'
assert not list(MODEL_DIR.glob('*.gguf')), 'GGUF is not supported by this notebook.'
print('checkpoint:', MODEL_DIR)

## 2. Baseline generation

Run this once before instrumentation. The output should be coherent; this establishes that the local checkpoint and tokenizer work.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

# IMPORTANT:
# Load the COMPLETE model into CPU RAM first.
# Do NOT call model.to(DEVICE) here.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32,
    local_files_only=True,
    low_cpu_mem_usage=True
).eval()


# ---------------------------------------------------------
# Move everything EXCEPT the expert banks to the GPU.
# ---------------------------------------------------------

if DEVICE.type == 'cuda':

    for name, param in model.named_parameters():
        # Expert weights stay in CPU RAM.
        if '.mlp.experts.' in name:
            continue

        param.data = param.data.to(DEVICE)

    for name, buffer in model.named_buffers():
        # Expert buffers, if any, stay in CPU RAM.
        if '.mlp.experts.' in name:
            continue

        buffer.data = buffer.data.to(DEVICE)


# ---------------------------------------------------------
# Verify placement
# ---------------------------------------------------------

print("\nDevice placement:")

for name, param in model.named_parameters():
    if '.mlp.experts.' in name:
        print(f"CPU expert: {name}")
        break

print(f"Model device: {model.device}")

messages = [
    {
        'role': 'user',
        'content': PROMPT
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors='pt'
)

if torch.is_tensor(inputs):
    inputs = {'input_ids': inputs}
else:
    inputs = dict(inputs)

inputs = {
    key: value.to(DEVICE) if torch.is_tensor(value) else value
    for key, value in inputs.items()
}

input_ids = inputs['input_ids']

In [ ]:
free,total = torch.cuda.mem_get_info(device=DEVICE)
print(f'GPU memory: {free/1e9:.2f} GB free /{total*memory_fraction} GB total')

In [ ]:
'''with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
print(tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True))'''

## 3. Locate OLMoE expert modules

This introspection avoids hard-coding a guessed layer path. It prints the expert-containing modules so the tracing hook is attached to the actual Transformers implementation installed in the environment.

In [ ]:
expert_modules = [(name, module) for name, module in model.named_modules() if 'expert' in name.lower() or 'mlp' in name.lower()]
for name, module in expert_modules[:80]:
    print(name, type(module).__name__)
assert expert_modules, 'No expert/MLP modules found; check the Transformers version and model class.'

## 4. Router/expert tracing

The hook records expert module calls and tensor shapes. Depending on the installed Transformers version, OLMoE may execute experts through a fused dispatch path; if expert IDs are not exposed by module hooks, the next cell reports that explicitly rather than inventing IDs.

In [ ]:
@dataclass
class RuntimeSample:
    token_index: int
    elapsed_ms: float
    ram_gib: float
    vram_gib: float

class LazyOLMoEExpertCache:
    """CPU-backed, per-layer LRU cache for OLMoE's fused expert tensors."""
    def __init__(self, experts, device, capacity=8):
        self.experts = experts
        self.device = torch.device(device)
        self.capacity = max(1, int(capacity))
        self.cache = OrderedDict()
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.bytes_loaded = 0
        self.accesses = Counter()
        # Keep the complete checkpoint copy in CPU RAM; only selected slices
        # are materialized on the execution device.
        self.experts.to('cpu')
        self.num_experts = int(experts.num_experts)

    def _expert_bytes(self, expert_id):
        return (self.experts.gate_up_proj[expert_id].numel() * self.experts.gate_up_proj.element_size()
                + self.experts.down_proj[expert_id].numel() * self.experts.down_proj.element_size())

    def get(self, expert_id):
        expert_id = int(expert_id)
        self.accesses[expert_id] += 1
        if expert_id in self.cache:
            self.hits += 1
            weights = self.cache.pop(expert_id)
            self.cache[expert_id] = weights
            return weights
        self.misses += 1
        gate_up = self.experts.gate_up_proj[expert_id].detach().to(self.device)
        down = self.experts.down_proj[expert_id].detach().to(self.device)
        self.cache[expert_id] = (gate_up, down)
        self.bytes_loaded += self._expert_bytes(expert_id)
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)
            self.evictions += 1
        return gate_up, down

    def forward(self, hidden_states, top_k_index, top_k_weights):
        final_hidden_states = torch.zeros_like(hidden_states)
        expert_mask = torch.nn.functional.one_hot(top_k_index, num_classes=self.num_experts)
        expert_mask = expert_mask.permute(2, 1, 0)
        expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
        for expert_idx_tensor in expert_hit:
            expert_idx = int(expert_idx_tensor.item())
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            gate_up, down = self.get(expert_idx)
            current_state = hidden_states[token_idx]
            gate, up = torch.nn.functional.linear(current_state, gate_up).chunk(2, dim=-1)
            current_hidden_states = self.experts.act_fn(gate) * up
            current_hidden_states = torch.nn.functional.linear(current_hidden_states, down)
            current_hidden_states *= top_k_weights[token_idx, top_k_pos, None]
            final_hidden_states.index_add_(0, token_idx, current_hidden_states.to(final_hidden_states.dtype))
        return final_hidden_states

    def summary(self, layer_index):
        total = self.hits + self.misses
        return {
            'layer': layer_index,
            'capacity': self.capacity,
            'resident': len(self.cache),
            'accesses': total,
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': self.hits / total if total else 0.0,
            'evictions': self.evictions,
            'bytes_loaded': self.bytes_loaded,
            'unique_experts': len(self.accesses),
        }

CACHE_CAPACITY = 48
layers = getattr(getattr(model, 'model', model), 'layers', None)
if layers is None:
    raise RuntimeError('Could not locate model layers for OLMoE cache installation.')

layer_caches = {}
for layer_index, layer in enumerate(layers):
    moe = getattr(layer, 'mlp', None)
    experts = getattr(moe, 'experts', None)
    if moe is None or not all(hasattr(experts, name) for name in ('gate_up_proj', 'down_proj', 'act_fn')):
        continue
    cache = LazyOLMoEExpertCache(experts, DEVICE, capacity=CACHE_CAPACITY)
    layer_caches[layer_index] = cache
    experts.forward = cache.forward

if not layer_caches:
    raise RuntimeError('No fused OLMoE expert banks were found.')

samples = []
process = psutil.Process()
t0 = time.perf_counter()
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_dict_in_generate=True)
elapsed = (time.perf_counter() - t0) * 1000
samples.append(RuntimeSample(MAX_NEW_TOKENS, elapsed / MAX_NEW_TOKENS, process.memory_info().rss / 2**30, torch.cuda.memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0))
print(f'cached layers: {len(layer_caches)}; capacity per layer: {CACHE_CAPACITY}')
print(f'generated tokens: {out.sequences.shape[-1] - input_ids.shape[-1]}')
print(f'wall time: {elapsed:.1f} ms; mean generated-token time: {samples[0].elapsed_ms:.1f} ms')
cache_stats = pd.DataFrame([cache.summary(layer_index) for layer_index, cache in layer_caches.items()])
print(cache_stats[['layer', 'accesses', 'hits', 'misses', 'hit_rate', 'evictions']].to_string(index=False))

## 5. What this validates

At this point we have verified the real model can run locally and measured baseline memory/latency. The expert module listing and hook output determine the exact OLMoE dispatch API for the next implementation step: replacing the routed expert bank with a disk-backed, fixed-capacity cache. Do not interpret `module_calls` as expert IDs unless the printed module names or outputs identify a specific expert.

The next cell is intentionally a gate: the lazy expert loader must be implemented against the observed module structure before collecting cache results.

In [ ]:
assert layer_caches, 'The lazy expert cache was not installed.'
assert all(row['accesses'] == row['hits'] + row['misses'] for row in (cache.summary(i) for i, cache in layer_caches.items()))
print('Lazy expert cache validation: PASS')
print(f'total expert calls: {sum(cache.hits + cache.misses for cache in layer_caches.values())}')
print(f'total cache hits: {sum(cache.hits for cache in layer_caches.values())}')
print(f'total cache misses: {sum(cache.misses for cache in layer_caches.values())}')

In [ ]:
pd.DataFrame(asdict(x) for x in samples)